# Image Stitching Step-by-Step

In this notebook, we break down the image stitching process and visualize the intermediate results. We also compare simple binary blending with advanced Laplacian pyramid blending.

In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Add the src directory to Python path
src_path = str(Path('..') / 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

from image_stitching.stitch_toy import TwoImageToyStitcher

stitcher = TwoImageToyStitcher()
img1_path = '../imgs/1.jpg'
img2_path = '../imgs/2.jpg'
stitcher._read(img1_path, img2_path)
image1, image2, image1Gray, image2Gray = stitcher.image1, stitcher.image2, stitcher.image1Gray, stitcher.image2Gray

## 1. Registering & Warping

In [ ]:
pts1, des1, pts2, des2 = stitcher._computeDescription(image1Gray, image2Gray)
matchedPts1, matchedPts2 = stitcher._matchDescription(pts1, des1, pts2, des2)
H = stitcher._computeHomography(matchedPts1, matchedPts2)
warpedImage1, warpedImage2, mask1, mask2 = stitcher._warp(image1, image2, H)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
ax1.imshow(cv2.cvtColor(warpedImage1.astype(np.uint8), cv2.COLOR_BGR2RGB))
ax1.set_title("Warped Image 1")
ax2.imshow(cv2.cvtColor(warpedImage2.astype(np.uint8), cv2.COLOR_BGR2RGB))
ax2.set_title("Warped Image 2")
plt.show()

## 2. Comparison: Simple vs. Advanced Blending

In [ ]:
# 1. Create a binary weight mask using Distance Transform
dist1 = cv2.distanceTransform(mask1, cv2.DIST_L2, 3)
dist2 = cv2.distanceTransform(mask2, cv2.DIST_L2, 3)
weight_mask = (dist1 > dist2).astype(np.float32)

# 2. Simple Blend (Binary Seam)
simple_result = stitcher.blend(warpedImage1, warpedImage2, weight_mask)

# 3. Advanced Blend (Laplacian Pyramid)
advanced_result = stitcher.softBlend(warpedImage1, warpedImage2, mask1, mask2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
ax1.imshow(cv2.cvtColor(simple_result.astype(np.uint8), cv2.COLOR_BGR2RGB))
ax1.set_title("Simple Binary Blending")
ax1.axis('off')

ax2.imshow(cv2.cvtColor(np.clip(advanced_result, 0, 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
ax2.set_title("Advanced Laplacian Blending")
ax2.axis('off')

plt.show()